# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier.

**Why it fits:** Our task is a "which first?" ranking task (generating a queue of pages to review). A Random Forest outputs probabilities which we can use to rank the pages. It handles non-linear relationships well (like specific impressions vs. position thresholds), and doesn't require aggressive feature scaling. We evaluate it using Precision@K, exactly like the baseline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
import matplotlib.pyplot as plt

## 2. Split design

**Design:** 5-Fold Grouped Cross-Validation (GroupKFold by `client_hash_id`).

**Why it's honest:** Since we are using the 30-day sealed dataset we cached in Week 4 (`baseline_action_score.csv`), we MUST prevent leakage between clients. Pages on the same client's domain share seasonality, authority, and ranking updates. A random split would leak this client-level context. Grouping by client ensures the model is tested on clients it has never seen during training.

In [2]:
# 1. Load the cached dataset from Week 4
df = pd.read_csv('../outputs/baseline_action_score.csv')
df['pos_change'] = df['pos_second_half'] - df['pos_first_half']

# 2. Setup Features and Target
features = ['imp_past15', 'pos_first_half', 'pos_second_half', 'pos_change']
X = df[features]
y = df['dropped_traffic_next15d']
groups = df['client_hash_id']

# 3. GroupKFold Cross Validation
gkf = GroupKFold(n_splits=5)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

oof_preds = np.zeros(len(df))

for train_idx, val_idx in gkf.split(X, y, groups):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model.fit(X_train, y_train)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

df['model_pred_prob'] = oof_preds


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# 1. Helper function for Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 2. Compare Baseline vs Model
results = []
base_rate = y.mean()
for k in [20, 50, 100]:
    base_p = precision_at_k(df['score'], y, k)
    model_p = precision_at_k(df['model_pred_prob'], y, k)
    results.append({'K': k, 'Base Rate': base_rate, 'Baseline P@K': base_p, 'Model P@K': model_p})

comparison_df = pd.DataFrame(results)
display(comparison_df)


,K,Base Rate,Baseline P@K,Model P@K
0,20,0.518897,0.65,0.30
1,50,0.518897,0.74,0.34
2,100,0.518897,0.75,0.40


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# 1. Feature Importances (from the last fitted fold)
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("--- Feature Importances ---")
print(importances.round(4))
print("\n")

# 2. Error Analysis: Let's look at the Top predicted by the model that were WRONG (False Positives)
top_model_picks = df.sort_values('model_pred_prob', ascending=False).head(50)
errors = top_model_picks[top_model_picks['dropped_traffic_next15d'] == 0]
print("--- Sample of Model Errors (False Positives in Top 50) ---")
display(errors[['client_hash_id', 'imp_past15', 'pos_first_half', 'pos_second_half', 'pos_change', 'model_pred_prob']].head(5))


--- Feature Importances ---
pos_second_half    0.3082
pos_change         0.2679
pos_first_half     0.2642
imp_past15         0.1597
dtype: float64


--- Sample of Model Errors (False Positives in Top 50) ---


,client_hash_id,imp_past15,pos_first_half,pos_second_half,pos_change,model_pred_prob
11,client_e547b89c05043229,45510.0,39.322652,42.842588,3.519936,0.770763
2639,client_e547b89c05043229,1545.0,40.341995,42.806149,2.464154,0.769019
3386,client_fef1a8f436438636,1255.0,40.861350,43.080336,2.218986,0.769019
2124,client_e547b89c05043229,1856.0,27.398358,29.113451,1.715094,0.768743
3368,client_e547b89c05043229,1262.0,30.441885,32.737196,2.295310,0.768743


**Interpretation:**
*   **Performance:** The baseline business logic absolutely crushes the Random Forest model! The baseline achieves 75% precision in the top 100, while the ML model struggles at 40%. The added complexity of the ML model actually hurt performance on unseen clients.
*   **What it leans on:** The model leans heaviest on absolute rank positions (`pos_second_half` at 0.308, `pos_first_half` at 0.264). This is a classic ML trap: absolute position means different things for different clients. The baseline only looked at the *relative* rank drop, which generalizes much better.
*   **Where it's wrong:** The model failed because it overfit to the specific rank positions of the training clients and failed on the holdout clients (GroupKFold). Additionally, because the model outputs bounded probabilities [0,1], it fails to float the absolute highest-impact pages to the top of the queue the way the baseline's raw impression multiplier did.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.